In [18]:
import pandas as pd
import numpy as np

In [19]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Chandni Chowk, Delhi - IITM.xlsx",skiprows=16)

In [20]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (645, 22)


In [21]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 331
Missing values after imputation:
 From Date      0
To Date        0
PM2.5          0
PM10           0
NO             0
NO2            0
NOx            0
NH3            0
SO2            0
CO             0
Ozone          0
Benzene        0
Toluene        0
Eth-Benzene    0
MP-Xylene      0
RH             0
WS             0
WD             0
BP             0
Xylene         0
AT             0
RF             0
dtype: int64


C:\Users\samru\AppData\Local\Temp\ipykernel_15996\3244313425.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(mode_val[0])


In [22]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

In [23]:
# ---------- 5. Convert date columns to datetime ----------
if 'From Date' in df.columns:
    df['From Date'] = pd.to_datetime(df['From Date'], errors='coerce')
if 'To Date' in df.columns:
    df['To Date'] = pd.to_datetime(df['To Date'], errors='coerce')

In [24]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (314, 22)
   From Date    To Date   PM2.5    PM10     NO    NO2    NOx    NH3    SO2  \
0 2025-01-01 2025-02-01  119.89  225.22  15.55   5.95  50.53  36.19  22.00   
1 2025-02-01 2025-03-01   26.62  129.32  15.55  45.33  37.88  36.19  14.69   
2 2025-03-01 2025-04-01  221.84  129.32  15.55   6.73  35.12  44.61  19.88   
3 2025-04-01 2025-05-01  167.89  240.65  15.55   4.04  37.80  50.02  20.34   
4 2025-05-01 2025-06-01   96.10  235.84  32.03  71.09  63.88  36.19  23.11   

     CO  ...  Toluene  Eth-Benzene  MP-Xylene     RH    WS       WD     BP  \
0  0.84  ...     7.46         2.36       1.56  29.03  1.23  187.855  996.5   
1  1.27  ...     7.46         2.38       1.56  29.43  1.23  187.855  996.5   
2  1.38  ...     7.23         2.80       2.07  29.93  1.23  187.855  996.5   
3  1.00  ...     7.45         2.04       1.61  29.93  1.23  187.855  996.5   
4  1.38  ...     7.45         2.04       1.61  29.93  1.23  187.855  996.5   

   Xylene     AT   RF  
0    2.79  29.5

In [25]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [26]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,...,Toluene,Eth-Benzene,MP-Xylene,RH,WS,WD,BP,Xylene,AT,RF
0,2025-01-01,2025-02-01,0.687218,1.087201,-0.206359,-1.802721,0.470353,0.047489,1.392654,-1.093882,...,0.102257,0.392741,-0.301992,-1.428096,0.115144,-0.098147,0.760663,0.976749,0.255874,0.0
1,2025-02-01,2025-03-01,-0.891286,-0.177313,-0.206359,-0.108014,-0.185157,0.047489,-0.047126,-0.168759,...,0.102257,0.411443,-0.301992,-1.409603,0.115144,-0.098147,0.760663,0.994326,-2.794699,0.0
2,2025-03-01,2025-04-01,2.412623,-0.177313,-0.206359,-1.769154,-0.328178,0.802700,0.975098,0.067901,...,0.045344,0.804184,-0.024470,-1.386486,0.115144,-0.098147,0.760663,2.444462,0.355317,0.0
3,2025-04-01,2025-05-01,1.499572,1.290657,-0.206359,-1.884918,-0.189303,1.287937,1.065700,-0.749650,...,0.099783,0.093510,-0.274784,-1.386486,0.115144,-0.098147,0.760663,-0.262457,0.355317,0.0
4,2025-05-01,2025-06-01,0.284596,1.227234,1.234947,1.000560,1.162137,0.047489,1.611280,0.067901,...,0.099783,0.093510,-0.274784,-1.386486,0.115144,-0.098147,0.760663,-0.262457,0.355317,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
309,2025-12-11,NaT,5.536121,-0.177313,-0.366407,-0.638632,-0.681065,2.217152,0.281798,1.746031,...,0.094834,-0.710674,0.193195,-0.607926,0.691577,0.988996,-0.262778,-0.833723,-2.005002,0.0
310,NaT,NaT,4.190154,-0.177313,-0.448617,-0.383437,-0.552036,2.146295,1.447803,1.423314,...,0.151746,-0.673271,0.236727,-0.606077,-0.397241,0.829443,-0.357321,-0.816145,0.255874,0.0
311,NaT,NaT,2.421762,-0.177313,-0.344542,-0.185047,-0.448398,0.047489,1.276448,0.993024,...,0.151746,-0.673271,0.236727,-0.583885,-0.109025,1.239965,-0.350709,-0.824934,0.255874,0.0
312,NaT,NaT,2.487089,-0.177313,-1.126416,-1.260054,-1.591526,0.630490,0.931767,1.616944,...,0.151746,-0.673271,0.236727,-0.633816,0.211216,1.017140,-0.372527,-0.816145,0.255874,0.0


In [27]:
df.to_excel('chandanichowk2025.xlsx', index=False)